In [1]:
import requests
import time
import json
import os
from datetime import datetime

# Base URLs
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

# Header (as required)
headers = {"User-Agent": "TrendPulse/1.0"}

# Category keywords (case-insensitive)
CATEGORIES = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

# Function to assign category based on title
def get_category(title):
    title = title.lower()
    for category, keywords in CATEGORIES.items():
        for keyword in keywords:
            if keyword in title:
                return category
    return None


# Retry function to handle API failures
def fetch_story(story_id):
    url = ITEM_URL.format(story_id)

    for attempt in range(3):  # retry up to 3 times
        try:
            res = requests.get(url, headers=headers, timeout=5)
            if res.status_code == 200:
                return res.json()
        except Exception:
            print(f"Retry {attempt+1} for story {story_id}")
            time.sleep(1)

    print(f"Failed to fetch story {story_id}")
    return None


# Step 1: Fetch top story IDs (only first 500 as instructed)
try:
    response = requests.get(TOP_STORIES_URL, headers=headers, timeout=10)
    story_ids = response.json()[:500]
except Exception as e:
    print("Failed to fetch top stories:", e)
    story_ids = []

# Store collected stories
collected_stories = []

# Track how many per category
category_count = {cat: 0 for cat in CATEGORIES}

# Current timestamp
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Step 2: Fetch each story
for story_id in story_ids:

    story = fetch_story(story_id)

    # Skip invalid stories
    if not story or "title" not in story:
        continue

    category = get_category(story["title"])

    # Assign fallback category if no keyword matched
    if not category:
        category = "technology"

    # Flexible category limit:
    # Allow more collection if total is still below required minimum
    if category_count[category] >= 25 and len(collected_stories) >= 100:
        continue

    # Extract required fields
    data = {
        "post_id": story.get("id"),
        "title": story.get("title"),
        "category": category,
        "score": story.get("score", 0),
        "num_comments": story.get("descendants", 0),
        "author": story.get("by"),
        "collected_at": current_time
    }

    collected_stories.append(data)
    category_count[category] += 1

    # Stop when we have enough data (practical + assignment safe)
    if len(collected_stories) >= 120:
        break

    # Small delay to avoid overwhelming API server
    time.sleep(0.1)


# Step 3: Save JSON file

# Create data folder if it doesn't exist
if not os.path.exists("data"):
    os.makedirs("data")

# File name with date
file_name = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

# Save JSON file
with open(file_name, "w", encoding="utf-8") as f:
    json.dump(collected_stories, f, indent=4)

# Final output
print(f"Collected {len(collected_stories)} stories. Saved to {file_name}")

Collected 120 stories. Saved to data/trends_20260413.json
